In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

try:
    from unidecode import unidecode
except ModuleNotFoundError:
    !pip install unidecode
    from unidecode import unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 8.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
from google.colab import files

# Upload:
# 1. processed_icd10_reference_valid_only.csv
# 2. leaderboard_data.csv
# 3. safe_gem_literal_map.csv
uploaded = files.upload()

print("Uploaded files:")
for name in uploaded.keys():
    print(name)

icd10_file = None
test_file = None
safe_gem_file = None

for name in uploaded.keys():
    lower = name.lower()

    if "processed_icd10_reference_valid_only" in lower or "icd10" in lower:
        icd10_file = name

    elif "leaderboard" in lower or "test" in lower:
        test_file = name

    elif "safe_gem_literal_map" in lower or "gem" in lower:
        safe_gem_file = name

print("Detected ICD-10 file:", icd10_file)
print("Detected leaderboard/test file:", test_file)
print("Detected safe GEM map file:", safe_gem_file)

if icd10_file is None:
    raise ValueError("Please upload processed_icd10_reference_valid_only.csv")

if test_file is None:
    raise ValueError("Please upload leaderboard_data.csv")

if safe_gem_file is None:
    raise ValueError("Please upload safe_gem_literal_map.csv")

icd10_df = pd.read_csv(icd10_file)
test_df = pd.read_csv(test_file)
safe_gem_map_df = pd.read_csv(safe_gem_file)

print("icd10_df:", icd10_df.shape)
print("test_df:", test_df.shape)
print("safe_gem_map_df:", safe_gem_map_df.shape)

display(icd10_df.head())
display(test_df.head())
display(safe_gem_map_df.head())

Saving safe_gem_literal_map.csv to safe_gem_literal_map.csv
Saving leaderboard_data (1).csv to leaderboard_data (1).csv
Saving processed_icd10_reference_valid_only (1).csv to processed_icd10_reference_valid_only (1).csv
Uploaded files:
safe_gem_literal_map.csv
leaderboard_data (1).csv
processed_icd10_reference_valid_only (1).csv
Detected ICD-10 file: processed_icd10_reference_valid_only (1).csv
Detected leaderboard/test file: leaderboard_data (1).csv
Detected safe GEM map file: safe_gem_literal_map.csv
icd10_df: (9943, 10)
test_df: (6667, 2)
safe_gem_map_df: (281, 5)


,Code,Literal,Code_clean,Literal_basic,in_icd_reference,D_P,Description,y_category,y_full_code,Literal_match
0,J9809,Hiperreactividad bronquial,J9809,Hiperreactividad bronquial,True,D,"Otras enfermedades de los bronquios, no clasif...",J,J9809,hiperreactividad bronquial
1,J9801,broncoespástica,J9801,broncoespástica,True,D,Broncoespasmo agudo,J,J9801,broncoespastica
2,I420,miocardiopatía dilatada,I420,miocardiopatía dilatada,True,D,Miocardiopatía dilatada,I,I420,miocardiopatia dilatada
3,Y831,HTA irc 6,Y831,HTA irc 6,True,D,Cirugía con implante de dispositivo interno ar...,Y,Y831,hta irc 6
4,R5600,Crisis febriles atípicas,R5600,Crisis febriles atípicas,True,D,Convulsiones febriles simples,R,R5600,crisis febriles atipicas


,id,Literal
0,1,AMNIODRENAJE
1,2,Hiperparatiroidismo primario
2,3,MIGRANYA parto
3,4,VHC
4,5,Absceso mama izq


,Literal_match,mapped_category,count,total_count,purity
0,a.e.,O,1,1,1.0
1,absceso retrofaringeo,J,2,2,1.0
2,acalasia,K,1,1,1.0
3,accidente,X,1,1,1.0
4,aciclovir,T,1,1,1.0


In [3]:
def basic_normalise_text(text):
    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.replace("´", "'").replace("`", "'").replace("’", "'")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_for_matching(text):
    text = str(text).lower()
    text = unidecode(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s+/.-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

if "Literal_match" not in icd10_df.columns:
    icd10_df["Literal_basic"] = icd10_df["Literal"].apply(basic_normalise_text)
    icd10_df["Literal_match"] = icd10_df["Literal_basic"].apply(normalize_for_matching)

if "Literal_match" not in test_df.columns:
    test_df["Literal_basic"] = test_df["Literal"].apply(basic_normalise_text)
    test_df["Literal_match"] = test_df["Literal_basic"].apply(normalize_for_matching)

if "y_category" not in icd10_df.columns:
    icd10_df["y_category"] = icd10_df["Code_clean"].astype(str).str[0]

In [4]:
train_icd10 = icd10_df.dropna(subset=["Literal_match", "y_category"]).copy()
train_icd10 = train_icd10[train_icd10["Literal_match"].astype(str).str.len() > 0].copy()

X_train_all = train_icd10["Literal_match"].fillna("").astype(str)
y_train_all = train_icd10["y_category"].astype(str)

final_svc = Pipeline([
    ("features", FeatureUnion([
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        )),
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        ))
    ])),
    ("clf", LinearSVC(
        C=0.5,
        class_weight=None,
        max_iter=20000,
        random_state=42
    ))
])

final_svc.fit(X_train_all, y_train_all)

test_df["svc_pred"] = final_svc.predict(
    test_df["Literal_match"].fillna("").astype(str)
)

In [5]:
#selctive context in svc
# ============================================================
# Selective context override
# Used after final_svc creates test_df["svc_pred"]
# ============================================================

def selective_context_override_text(text, pred):
    text = str(text).lower()
    pred = str(pred)

    # These are the conservative context patterns that generalised best.
    # They are intentionally narrow to avoid overfitting.

    # Puerperium / postpartum context usually belongs to obstetric category O.
    if ("puerperio" in text) or ("puerperi" in text):
        return "O"

    # Fetal context also usually belongs to obstetric/perinatal handling.
    # In our best submission this was treated as O.
    if ("fetal" in text) or ("feto" in text):
        return "O"

    return pred

test_df["selective_context_pred"] = [
    selective_context_override_text(text, pred)
    for text, pred in zip(test_df["Literal_match"], test_df["svc_pred"])
]

print("Selective context changes:")
print((test_df["selective_context_pred"] != test_df["svc_pred"]).sum())

display(test_df[
    test_df["selective_context_pred"] != test_df["svc_pred"]
][[
    "id", "Literal", "Literal_match", "svc_pred", "selective_context_pred"
]].head(100))

Selective context changes:
6


,id,Literal,Literal_match,svc_pred,selective_context_pred
371,372,coagulación laser feto,coagulacion laser feto,Z,O
1821,1822,feto,feto,Z,O
2564,2565,Ecografía : Feto,ecografia feto,B,O
2933,2934,Monitorizacion fetal,monitorizacion fetal,4,O
3891,3892,hipotiroidismo puerperio,hipotiroidismo puerperio,E,O
5803,5804,Monitorizacion fetal,monitorizacion fetal,4,O


In [6]:
official_gem_literal_to_cat = dict(zip(
    safe_gem_map_df["Literal_match"],
    safe_gem_map_df["mapped_category"]
))

safe_gem_categories = set(["B", "C", "D", "E", "G", "I", "J", "K", "N", "Q", "R"])

def has_risky_obstetric_context(text):
    text = str(text).lower()
    risky_terms = [
        "parto", "part", "puerperio", "puerperi", "gestacion",
        "embarazo", "fetal", "feto", "cesarea", "amnio"
    ]
    return any(re.search(rf"\b{re.escape(t)}\b", text) for t in risky_terms)

test_df["safe_gem_candidate"] = test_df["Literal_match"].map(official_gem_literal_to_cat)

test_df["use_safe_gem"] = (
    test_df["safe_gem_candidate"].notna() &
    test_df["safe_gem_candidate"].isin(safe_gem_categories) &
    ~test_df["Literal_match"].apply(has_risky_obstetric_context) &
    (test_df["safe_gem_candidate"] != test_df["selective_context_pred"])
)

test_df["final_pred"] = np.where(
    test_df["use_safe_gem"],
    test_df["safe_gem_candidate"],
    test_df["selective_context_pred"]
)

print("Safe GEM overrides used:", test_df["use_safe_gem"].sum())
print(test_df["final_pred"].value_counts())

Safe GEM overrides used: 53
final_pred
O    1033
Z     955
0     764
N     378
B     362
E     321
K     297
I     289
D     280
3     256
R     234
C     229
F     200
M     186
J     186
G     118
Q      88
1      78
H      73
V      72
L      72
P      53
S      36
T      33
4      17
Y      14
5      12
W       9
A       9
X       6
U       2
6       2
2       2
8       1
Name: count, dtype: int64


In [7]:
official_gem_literal_to_cat = dict(zip(
    safe_gem_map_df["Literal_match"],
    safe_gem_map_df["mapped_category"]
))

safe_gem_categories = set(["B", "C", "D", "E", "G", "I", "J", "K", "N", "Q", "R"])

def has_risky_obstetric_context(text):
    text = str(text).lower()
    risky_terms = [
        "parto", "part", "puerperio", "puerperi", "gestacion",
        "embarazo", "fetal", "feto", "cesarea", "amnio"
    ]
    return any(re.search(rf"\b{re.escape(t)}\b", text) for t in risky_terms)

test_df["safe_gem_candidate"] = test_df["Literal_match"].map(official_gem_literal_to_cat)

test_df["use_safe_gem"] = (
    test_df["safe_gem_candidate"].notna() &
    test_df["safe_gem_candidate"].isin(safe_gem_categories) &
    ~test_df["Literal_match"].apply(has_risky_obstetric_context) &
    (test_df["safe_gem_candidate"] != test_df["selective_context_pred"])
)

test_df["final_pred"] = np.where(
    test_df["use_safe_gem"],
    test_df["safe_gem_candidate"],
    test_df["selective_context_pred"]
)

print("Safe GEM overrides used:", test_df["use_safe_gem"].sum())
print(test_df["final_pred"].value_counts())

Safe GEM overrides used: 53
final_pred
O    1033
Z     955
0     764
N     378
B     362
E     321
K     297
I     289
D     280
3     256
R     234
C     229
F     200
M     186
J     186
G     118
Q      88
1      78
H      73
V      72
L      72
P      53
S      36
T      33
4      17
Y      14
5      12
W       9
A       9
X       6
U       2
6       2
2       2
8       1
Name: count, dtype: int64


In [9]:
submission = test_df[["id", "Literal", "final_pred"]].copy()
submission = submission.rename(columns={"final_pred": "y_category"})

print(submission.shape)
print(submission.isna().sum())
display(submission.head())

assert submission.shape[0] == 6667
assert submission.columns.tolist() == ["id", "Literal", "y_category"]
assert submission["y_category"].isna().sum() == 0

import os
from google.colab import files

os.makedirs("outputs", exist_ok=True)

submission.to_csv("outputs/final_submission.csv", index=False)

files.download("outputs/final_submission.csv")

(6667, 3)
id            0
Literal       0
y_category    0
dtype: int64


,id,Literal,y_category
0,1,AMNIODRENAJE,1
1,2,Hiperparatiroidismo primario,E
2,3,MIGRANYA parto,O
3,4,VHC,B
4,5,Absceso mama izq,N


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>